In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd

Load postings.csv from the team Drive copy (version 13). Check: 123,849 rows, 31 columns, 29.1% of postings have pay, matching DATA_MANIFEST.md.

In [2]:
drive.mount("/content/drive")
root = next(Path("/content/drive/MyDrive").glob("AI Job Market Skill Miner*"), None)
print("Team folder:", root)   # None means the shortcut is missing

raw = root / "data/raw/linkedin_postings_2023_24_v13"
print(sorted(p.name for p in raw.iterdir()))

df = pd.read_csv(raw / "postings.csv")
print("postings shape:", df.shape)

pay_cols = ["min_salary", "med_salary", "max_salary"]
has_pay = df[pay_cols].notna().any(axis=1)
print(f"with any pay: {has_pay.sum()} ({has_pay.mean():.1%})")

Mounted at /content/drive
Team folder: /content/drive/MyDrive/AI Job Market Skill Miner.
['README.txt', 'companies', 'jobs', 'mappings', 'postings.csv']
postings shape: (123849, 31)
with any pay: 36073 (29.1%)


Preliminary data/AI filter: keep postings whose title matches data/AI keywords, drop "data entry", "clerk" and "sales". Result: 1,772 postings, of which 583 show pay and 279 are entry level. Rija will refine the filter.

In [3]:
title = df["title"].fillna("").str.lower()
include = r"data scien|data analy|data engineer|machine learning|\bml\b|\bai\b|artificial intelligence|business intelligence|deep learning|nlp"
exclude = r"data entry|clerk|sales"
sub = df[title.str.contains(include, regex=True) & ~title.str.contains(exclude, regex=True)]
print("data/AI postings:", len(sub))
print("with pay:", sub[["min_salary","med_salary","max_salary"]].notna().any(axis=1).sum())
print("entry level:", (sub["formatted_experience_level"] == "Entry level").sum())

data/AI postings: 1772
with pay: 583
entry level: 279
